In [2]:
import torch
import numpy as np
from Dataloader import Dataset, Load_Dataset_A, Load_Dataset_B, Load_Dataset_C
from model import fNIRS_T, fNIRS_PreT
from LOSO_Split import Split_Dataset_A, Split_Dataset_B, Split_Dataset_C
from Parameters import parameters
import os


class LabelSmoothing(torch.nn.Module):
    """NLL loss with label smoothing."""
    def __init__(self, smoothing=0.0):
        super(LabelSmoothing, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing

    def forward(self, x, target):
        logprobs = torch.nn.functional.log_softmax(x, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


if __name__ == "__main__":
    # Training epochs
    EPOCH = 120

    data_path, dataset, model = parameters()

    # Save file and avoid training file overwriting.
    save_path = 'Results/' + dataset + '/LOSO/' + model
    assert os.path.exists(save_path) is False, 'path is exist'
    os.makedirs(save_path)

    # Load dataset, set flooding levels and number of Subjects. Different models may have different flooding levels.
    if dataset == 'A':
        flooding_level = [0, 0, 0]
        Subjects = 8
        if model == 'fNIRS-T':
            feature, label = Load_Dataset_A(data_path, model='fNIRS-T')
        elif model == 'fNIRS-PreT':
            feature, label = Load_Dataset_A(data_path, model='fNIRS-PreT')
    elif dataset == 'B':
        if model == 'fNIRS-T':
            flooding_level = [0.45, 0.40, 0.35]
        else:
            flooding_level = [0.40, 0.38, 0.35]
        Subjects = 29
        feature, label = Load_Dataset_B(data_path)
    elif dataset == 'C':
        flooding_level = [0.45, 0.40, 0.35]
        Subjects = 30
        feature, label = Load_Dataset_C(data_path)

    _, _, channels, sampling_points = feature.shape

    for sub in range(1, Subjects+1):
        # Spilt dataset, one subject's data is test set, and the rest is training set.
        if dataset == 'A':
            X_train, y_train, X_test, y_test = Split_Dataset_A(sub, feature, label, channels)
        elif dataset == 'B':
            X_train, y_train, X_test, y_test = Split_Dataset_B(sub, feature, label, channels)
        elif dataset == 'C':
            X_train, y_train, X_test, y_test = Split_Dataset_C(sub, feature, label, channels)

        path = save_path + '/' + str(sub)
        assert os.path.exists(path) is False, 'sub-path is exist'
        os.makedirs(path)

        train_set = Dataset(X_train, y_train, transform=True)
        test_set = Dataset(X_test, y_test, transform=True)
        train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
        test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=True)

        # -------------------------------------------------------------------------------------------------------------------- #
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        if dataset == 'A':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
        elif dataset == 'B':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=2, sampling_point=sampling_points, dim=64, depth=6, heads=8, mlp_dim=64).to(device)
        elif dataset == 'C':
            if model == 'fNIRS-T':
                net = fNIRS_T(n_class=3, sampling_point=sampling_points, dim=128, depth=6, heads=8, mlp_dim=64).to(device)
            elif model == 'fNIRS-PreT':
                net = fNIRS_PreT(n_class=3, sampling_point=sampling_points, dim=128, depth=6, heads=8, mlp_dim=64).to(device)

        criterion = LabelSmoothing(0.1)
        optimizer = torch.optim.AdamW(net.parameters())
        lrStep = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
        # -------------------------------------------------------------------------------------------------------------------- #
        test_max_acc = 0
        for epoch in range(EPOCH):
            net.train()
            train_running_acc = 0
            total = 0
            loss_steps = []
            for i, data in enumerate(train_loader):
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                loss = criterion(outputs, labels.long())

                # Piecewise decay flooding. b is flooding level, b = 0 means no flooding
                if epoch < 30:
                    b = flooding_level[0]
                elif epoch < 50:
                    b = flooding_level[1]
                else:
                    b = flooding_level[2]

                # flooding
                loss = (loss - b).abs() + b
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                loss_steps.append(loss.item())
                total += labels.shape[0]
                pred = outputs.argmax(dim=1, keepdim=True)
                train_running_acc += pred.eq(labels.view_as(pred)).sum().item()

            train_running_loss = float(np.mean(loss_steps))
            train_running_acc = 100 * train_running_acc / total
            print('[%d, %d] Train loss: %0.2f' % (sub, epoch, train_running_loss))
            print('[%d, %d] Train acc: %0.2f%%' % (sub, epoch, train_running_acc))

            # -------------------------------------------------------------------------------------------------------------------- #
            net.eval()
            test_running_acc = 0
            total = 0
            loss_steps = []
            with torch.no_grad():
                for data in test_loader:
                    inputs, labels = data
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    outputs = net(inputs)
                    loss = criterion(outputs, labels.long())

                    loss_steps.append(loss.item())
                    total += labels.shape[0]
                    pred = outputs.argmax(dim=1, keepdim=True)
                    test_running_acc += pred.eq(labels.view_as(pred)).sum().item()

                test_running_acc = 100 * test_running_acc / total
                test_running_loss = float(np.mean(loss_steps))
                print('[%d, %d] Test loss: %0.2f' % (sub, epoch, test_running_loss))
                print('[%d, %d] Test acc: %0.2f%%' % (sub, epoch, test_running_acc))

                if test_running_acc > test_max_acc:
                    test_max_acc = test_running_acc
                    torch.save(net.state_dict(), path + '/model.pt')
                    test_save = open(path + '/test_acc.txt', "w")
                    test_save.write("sub=%d, best_acc= %.2f" % (sub, test_running_acc))
                    test_save.close()

            lrStep.step()

B fNIRS-T
1  OK:  (60, 2, 36, 200)
2  OK:  (120, 2, 36, 200)
3  OK:  (180, 2, 36, 200)
4  OK:  (240, 2, 36, 200)
5  OK:  (300, 2, 36, 200)
6  OK:  (360, 2, 36, 200)
7  OK:  (420, 2, 36, 200)
8  OK:  (480, 2, 36, 200)
9  OK:  (540, 2, 36, 200)
10  OK:  (600, 2, 36, 200)
11  OK:  (660, 2, 36, 200)
12  OK:  (720, 2, 36, 200)
13  OK:  (780, 2, 36, 200)
14  OK:  (840, 2, 36, 200)
15  OK:  (900, 2, 36, 200)
16  OK:  (960, 2, 36, 200)
17  OK:  (1020, 2, 36, 200)
18  OK:  (1080, 2, 36, 200)
19  OK:  (1140, 2, 36, 200)
20  OK:  (1200, 2, 36, 200)
21  OK:  (1260, 2, 36, 200)
22  OK:  (1320, 2, 36, 200)
23  OK:  (1380, 2, 36, 200)
24  OK:  (1440, 2, 36, 200)
25  OK:  (1500, 2, 36, 200)
26  OK:  (1560, 2, 36, 200)
27  OK:  (1620, 2, 36, 200)
28  OK:  (1680, 2, 36, 200)
29  OK:  (1740, 2, 36, 200)
feature  (1740, 2, 36, 200)
label  (1740,)
torch.Size([1680, 2, 36, 200])
torch.Size([1680])
torch.Size([60, 2, 36, 200])
torch.Size([60])
[1, 0] Train loss: 0.78
[1, 0] Train acc: 54.17%
[1, 0] Test loss